# Ch05 实验：Colored MNIST——颜色捷径 vs 数字形状

**经典基准**：将 MNIST 数字二分类（0–4 → 类别 0，5–9 → 类别 1），植入颜色捷径。

**捷径设计**：
- 训练集：类别 0 的数字 95% 染红色，类别 1 的数字 95% 染蓝色
- ID 测试集：颜色分布与训练集相同
- OOD 测试集：颜色**完全反转**（类别 0 → 蓝色，类别 1 → 红色）

**为什么这次会成功**：
- 颜色（R 通道 vs B 通道）：第一层卷积核就能区分，极易学习
- 数字形状（0–4 vs 5–9）：需要更多 epoch 才能掌握
- 早停策略：在模型来得及学会形状之前就停下，捷径是唯一依赖

**目录**：配置 → 加载 MNIST → 染色分集 → 模型 → 训练 → 捷径检测 → 泛化测试 → 显著性图 → 置信度 → Tilted ERM

## 1. 导入 & 配置

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import fetch_openml
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = ['Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

TRAIN_BIAS = 0.95
TEST_BIAS  = 0.95

N_TRAIN  = 2000
N_TEST   = 500

STOP_ACC   = 0.88
MAX_EPOCHS = 10

print(f'device: {DEVICE}')

## 2. 加载 MNIST 并染色

**首次运行**会从 OpenML 下载 MNIST（约 55 MB），之后自动读缓存。

**染色规则**：
- 类别 0（数字 0–4）：以 `class0_red_prob` 的概率染红（R 通道），否则染蓝（B 通道）
- 类别 1（数字 5–9）：以 `class1_blue_prob` 的概率染蓝，否则染红

**三个子集**：

| 子集 | class0_red_prob | class1_blue_prob | 说明 |
|------|----------------|-----------------|------|
| train | 95% | 95% | 颜色强烈预测类别 |
| test_id | 95% | 95% | 与训练集同分布 |
| test_ood | **0%** | **0%** | 颜色完全反转 |

Dataset 中额外记录每个样本的 `color`（0=红，1=蓝），用于后续捷径检测。

In [ ]:
print('加载 MNIST（首次运行需要下载，约 55 MB）...')
mnist = fetch_openml('mnist_784', version=1, as_frame=False, cache=True, parser='auto')
X_all = mnist.data.astype(np.float32) / 255.0   # (70000, 784)，值域 [0,1]
y_bin = (mnist.target.astype(int) >= 5).astype(int)  # 0–4→0，5–9→1
print(f'MNIST 加载完成，class 0: {(y_bin==0).sum()}，class 1: {(y_bin==1).sum()}')


def colorize(pixel_row, color):
    """
    pixel_row : (784,) float32，值域 [0,1]，MNIST 灰度像素
    color     : 0=红（写入 R 通道），1=蓝（写入 B 通道）
    返回      : (3, 28, 28) float32 tensor，活跃通道归一化到 [-1,1]，其余通道为 0
    """
    gray = ((pixel_row.reshape(28, 28) - 0.5) / 0.5)  # [-1, 1]
    rgb  = np.zeros((3, 28, 28), dtype=np.float32)
    rgb[0 if color == 0 else 2] = gray               # 红→R，蓝→B
    return torch.from_numpy(rgb)


def make_split(X, y, n_per_class, class0_red_prob, class1_blue_prob):
    """
    从 X/y 中采样并染色。
    额外返回 colors 数组，记录每个样本的颜色（0=红，1=蓝），用于捷径检测。
    """
    images, labels, colors = [], [], []
    for cls, color_if_match, color_if_mismatch, match_prob in [
        (0, 0, 1, class0_red_prob),   # 类别 0：match_prob 概率→红
        (1, 1, 0, class1_blue_prob),  # 类别 1：match_prob 概率→蓝
    ]:
        idx = np.where(y == cls)[0]
        chosen = rng.choice(idx, n_per_class, replace=False)
        for i in chosen:
            color = color_if_match if rng.random() < match_prob else color_if_mismatch
            images.append(colorize(X[i], color))
            labels.append(cls)
            colors.append(color)

    return (
        torch.stack(images),                           # (2*n, 3, 28, 28)
        torch.tensor(labels, dtype=torch.long),        # (2*n,)
        torch.tensor(colors, dtype=torch.long),        # (2*n,)
    )


# 训练集用前 60000 条，测试集用后 10000 条（官方划分）
X_tr, y_tr = X_all[:60000], y_bin[:60000]
X_te, y_te = X_all[60000:], y_bin[60000:]

print('生成染色数据集...')
train_X, train_y, train_c = make_split(X_tr, y_tr, N_TRAIN,
                                        class0_red_prob=TRAIN_BIAS,
                                        class1_blue_prob=TRAIN_BIAS)
id_X,    id_y,    id_c    = make_split(X_te, y_te, N_TEST,
                                        class0_red_prob=TEST_BIAS,
                                        class1_blue_prob=TEST_BIAS)
ood_X,   ood_y,   ood_c   = make_split(X_te, y_te, N_TEST,
                                        class0_red_prob=0.0,   # 颜色全反转
                                        class1_blue_prob=0.0)

print(f'  train: {len(train_y)} | test_id: {len(id_y)} | test_ood: {len(ood_y)}')

In [ ]:
def tensor_to_rgb(t):
    """(3,28,28) tensor [-1,1] → (28,28,3) uint8，用于 imshow"""
    img = t.permute(1, 2, 0).numpy()   # (28,28,3)
    img = np.clip((img + 1) / 2, 0, 1) # [-1,1] → [0,1]
    return img

# 展示：训练集前 4 张（红色类别0，蓝色类别1）+ OOD 前 4 张（颜色反转）
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
titles = ['训练集（红=类0，蓝=类1）', 'OOD 测试集（颜色完全反转）']

for row, (imgs, labels) in enumerate([(train_X, train_y), (ood_X, ood_y)]):
    axes[row, 0].set_ylabel(titles[row], fontsize=9)
    for col in range(8):
        axes[row, col].imshow(tensor_to_rgb(imgs[col]))
        axes[row, col].set_title(f'类别 {labels[col].item()}', fontsize=8)
        axes[row, col].axis('off')

plt.suptitle('左8列=训练集  右8列=OOD 测试集（同一张图换了颜色）', fontsize=10)
plt.tight_layout()
plt.show()

## 3. Dataset & DataLoader

In [ ]:
class ColoredMNISTDataset(Dataset):
    """内存数据集，返回 (image, label, color)"""
    def __init__(self, images, labels, colors):
        self.images = images   # (N, 3, 28, 28)
        self.labels = labels   # (N,) 类别 0/1
        self.colors = colors   # (N,) 颜色 0=红/1=蓝

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx], self.colors[idx]


train_dataset = ColoredMNISTDataset(train_X, train_y, train_c)
id_dataset    = ColoredMNISTDataset(id_X,    id_y,    id_c)
ood_dataset   = ColoredMNISTDataset(ood_X,   ood_y,   ood_c)

train_loader    = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_id_loader  = DataLoader(id_dataset,   batch_size=128, shuffle=False)
test_ood_loader = DataLoader(ood_dataset,  batch_size=128, shuffle=False)

print(f'train: {len(train_dataset)} | test_id: {len(id_dataset)} | test_ood: {len(ood_dataset)}')

## 4. 模型定义（ColorCNN）

输入 **3×28×28**（RGB 彩色 MNIST），2 层卷积 + 2 层全连接。

第一层卷积核只需要学会"哪个通道有信号"就能区分红/蓝颜色——这是捷径的来源。

In [ ]:
class ColorCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),  # 3×28×28 → 16×28×28
            nn.ReLU(),
            nn.MaxPool2d(2),                              # → 16×14×14
            nn.Conv2d(16, 32, kernel_size=3, padding=1), # → 32×14×14
            nn.ReLU(),
            nn.MaxPool2d(2),                              # → 32×7×7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = ColorCNN().to(DEVICE)
print(f'参数量: {sum(p.numel() for p in model.parameters()):,}')

## 5. 训练（标准 ERM）

早停阈值设为 **88%**——颜色捷径单独就能达到 ~95%，所以 88% 时模型主要靠颜色，还没来得及充分学习形状。

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
train_losses, train_accs = [], []

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_loss, correct, total = 0.0, 0, 0
    for x, y, _ in train_loader:          # Dataset 返回 (image, label, color)
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        total += len(y)

    avg_loss = epoch_loss / total
    acc = correct / total
    train_losses.append(avg_loss)
    train_accs.append(acc)
    print(f'Epoch {epoch:2d}  loss={avg_loss:.4f}  train_acc={acc:.1%}')

    if acc >= STOP_ACC:
        print(f'\n→ 准确率达到 {acc:.1%}（早停阈值 {STOP_ACC:.0%}），停止训练')
        print('  此时模型主要依赖颜色捷径，还未充分学会数字形状')
        break

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.plot(train_losses); ax1.set_title('训练 Loss'); ax1.set_xlabel('epoch')
ax2.plot(train_accs);   ax2.set_title('训练准确率'); ax2.set_xlabel('epoch')
ax2.axhline(STOP_ACC, color='r', linestyle='--', label=f'早停 {STOP_ACC:.0%}')
ax2.legend()
plt.tight_layout()
plt.show()

## 6. 捷径检测

**方法**：在 ID 测试集中，将样本分为两组：
- **颜色匹配**：颜色与类别一致（类别 0 → 红，类别 1 → 蓝）——捷径有效
- **颜色不匹配**：颜色与类别相反——捷径失效，只能靠形状

若两组准确率差距大 → 模型依赖颜色捷径；若差距小 → 模型靠形状。

In [ ]:
def shortcut_detection(model, dataset, device='cpu'):
    """
    颜色捷径检测：比较"颜色匹配"和"颜色不匹配"样本的准确率差距。
    - 颜色匹配：捷径有效（如类别 0 + 红色）
    - 颜色不匹配：捷径失效（如类别 0 + 蓝色），模型只能靠形状
    差距越大 → 模型越依赖颜色捷径。
    """
    model.eval()
    match_correct,   match_total   = 0, 0
    mismatch_correct, mismatch_total = 0, 0

    with torch.no_grad():
        for x, y, c in DataLoader(dataset, batch_size=128):
            x, y, c = x.to(device), y, c
            preds = model(x).argmax(1).cpu()

            # 颜色匹配：(类别0 且 红色) 或 (类别1 且 蓝色)
            matched = (y == c)      # class 0→red(0), class 1→blue(1) 时 y==c
            match_correct    += (preds[matched]  == y[matched]).sum().item()
            match_total      += matched.sum().item()
            mismatch_correct += (preds[~matched] == y[~matched]).sum().item()
            mismatch_total   += (~matched).sum().item()

    acc_match    = match_correct    / match_total    if match_total    else float('nan')
    acc_mismatch = mismatch_correct / mismatch_total if mismatch_total else float('nan')
    gap = acc_match - acc_mismatch

    print(f'颜色匹配样本   准确率: {acc_match:.1%}  (n={match_total})')
    print(f'颜色不匹配样本 准确率: {acc_mismatch:.1%}  (n={mismatch_total})')
    print(f'差距: {gap:.1%}')
    if gap > 0.3:
        print('→ 模型严重依赖颜色捷径（不匹配时准确率大幅下降）')
    elif gap > 0.1:
        print('→ 模型部分依赖颜色捷径')
    else:
        print('→ 模型主要靠形状，颜色影响小')
    return acc_match, acc_mismatch, gap


print('=== 捷径检测（ID 测试集）===')
shortcut_detection(model, id_dataset, device=DEVICE)

## 7. 泛化测试

In [ ]:
def generalization_test(model, id_loader, ood_loader, device='cpu'):
    def evaluate(loader):
        model.eval()
        correct, total, high_conf_wrong = 0, 0, 0
        with torch.no_grad():
            for x, y, _ in loader:
                x, y = x.to(device), y.to(device)
                probs = torch.softmax(model(x), dim=1)
                pred = probs.argmax(1)
                conf = probs.max(1).values
                correct += (pred == y).sum().item()
                high_conf_wrong += ((conf > 0.8) & (pred != y)).sum().item()
                total += y.size(0)
        return correct / total, high_conf_wrong / total

    acc_id,  _        = evaluate(id_loader)
    acc_ood, hcw_ood  = evaluate(ood_loader)
    gap = acc_id - acc_ood

    print(f'ID  准确率: {acc_id:.1%}')
    print(f'OOD 准确率: {acc_ood:.1%}')
    print(f'泛化差距:   {gap:.1%}')
    if gap < 0.05:
        print('→ 泛化良好，模型可能靠形状')
    elif gap < 0.3:
        print('→ 泛化中等，部分依赖颜色捷径')
    else:
        print('→ 泛化差，模型严重依赖颜色捷径')
    if hcw_ood > 0.1:
        print(f'⚠ OOD 高置信错误率 {hcw_ood:.1%}，存在过度自信')
    return acc_id, acc_ood, gap


print('=== ERM 模型泛化测试 ===')
acc_id, acc_ood, gap = generalization_test(model, test_id_loader, test_ood_loader, device=DEVICE)

## 8. 显著性图可视化

用梯度绝对值作为显著性。如果模型依赖颜色捷径，显著性会均匀分布在整个数字区域（颜色是通道级别的信号）；如果模型靠形状，显著性会集中在数字的轮廓和笔画上。

In [ ]:
def saliency_map(model, x_sample, target_class):
    """x_sample: (1, 3, 28, 28)；返回 (28, 28) 显著性图"""
    model.eval()
    x = x_sample.clone().detach().requires_grad_(True)
    model(x)[0, target_class].backward()
    sal = x.grad.data.abs().max(dim=1)[0]   # 跨通道取最大
    return sal.squeeze().numpy()


# 从 OOD 测试集取样展示（颜色已反转，捷径失效）
ood_iter = iter(test_ood_loader)
ood_x, ood_y, ood_c = next(ood_iter)

n_show = 4
fig, axes = plt.subplots(2, n_show, figsize=(12, 5))
CLASSES = ['0–4', '5–9']

for i in range(n_show):
    x0 = ood_x[i:i+1].to(DEVICE)
    true_label = ood_y[i].item()
    pred_label = model(x0).argmax(1).item()
    color_name = '红' if ood_c[i].item() == 0 else '蓝'

    sal = saliency_map(model, x0.cpu(), pred_label)

    axes[0, i].imshow(tensor_to_rgb(ood_x[i]))
    axes[0, i].set_title(
        f'真:{CLASSES[true_label]}({color_name})\n预:{CLASSES[pred_label]}', fontsize=8)
    axes[0, i].axis('off')

    axes[1, i].imshow(sal, cmap='hot')
    axes[1, i].set_title('显著性图', fontsize=8)
    axes[1, i].axis('off')

plt.suptitle('OOD 测试集显著性图\n颜色反转后，模型关注的是颜色通道还是数字轮廓？', fontsize=10)
plt.tight_layout()
plt.show()

## 9. 置信度校准分析

核心问题：模型在 OOD 数据上失败时，它知道自己在失败吗？

In [ ]:
def get_confidence_and_accuracy(model, dataloader, device='cpu'):
    model.eval()
    confidences, corrects = [], []
    with torch.no_grad():
        for x, y, _ in dataloader:
            probs = torch.softmax(model(x.to(device)), dim=1).cpu()
            conf, pred = probs.max(1)
            corrects.extend((pred == y).numpy())
            confidences.extend(conf.numpy())
    return np.array(confidences), np.array(corrects)


conf_id,  correct_id  = get_confidence_and_accuracy(model, test_id_loader,  DEVICE)
conf_ood, correct_ood = get_confidence_and_accuracy(model, test_ood_loader, DEVICE)

print(f'ID  数据 — 准确率: {correct_id.mean():.1%}，平均置信度: {conf_id.mean():.3f}')
print(f'OOD 数据 — 准确率: {correct_ood.mean():.1%}，平均置信度: {conf_ood.mean():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(conf_id,  bins=20, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title(f'ID 置信度分布\n准确率={correct_id.mean():.1%}', fontsize=10)
axes[0].set_xlabel('最大 softmax 概率'); axes[0].set_ylabel('样本数')
axes[1].hist(conf_ood, bins=20, color='tomato',    edgecolor='white', alpha=0.85)
axes[1].set_title(f'OOD 置信度分布\n准确率={correct_ood.mean():.1%}', fontsize=10)
axes[1].set_xlabel('最大 softmax 概率')
plt.suptitle('OOD 置信度高 + 准确率低 = 过度自信', fontsize=10)
plt.tight_layout(); plt.show()

if conf_ood.mean() > 0.8 and correct_ood.mean() < 0.6:
    print('⚠ 过度自信：模型在自信地犯错')
elif conf_ood.mean() < 0.6:
    print('✓ 模型知道自己不确定')
else:
    print('置信度适中，结合准确率综合判断')

## 10. 缓解实验：Tilted ERM

Tilted ERM 给高损失样本（被捷径害到的困难样本）更高权重，`t` 越大越激进。  
观察：泛化差距是否缩小？代价是什么？

In [ ]:
def tilted_erm_train(model, dataloader, t=5.0, lr=1e-3, epochs=10, device='cpu'):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(reduction='none')
    for epoch in range(1, epochs + 1):
        model.train()
        for x, y, _ in dataloader:
            x, y = x.to(device), y.to(device)
            losses = criterion(model(x), y)
            tilted_loss = (1.0 / t) * torch.log(
                (1.0 / len(losses)) * torch.exp(t * losses).sum()
            )
            optimizer.zero_grad()
            tilted_loss.backward()
            optimizer.step()
        if epoch % 5 == 0:
            print(f'  epoch {epoch}')
    return model


model_tilted = ColorCNN().to(DEVICE)
print('Tilted ERM 训练中（t=5，10 epochs）...')
model_tilted = tilted_erm_train(model_tilted, train_loader, t=5.0, epochs=10, device=DEVICE)

print('\n=== Tilted ERM 泛化测试 ===')
acc_id_t, acc_ood_t, gap_t = generalization_test(model_tilted, test_id_loader, test_ood_loader, DEVICE)

print(f'\n对比汇总：')
print(f'  标准 ERM   — ID: {acc_id:.1%}  OOD: {acc_ood:.1%}  gap: {gap:.1%}')
print(f'  Tilted ERM — ID: {acc_id_t:.1%}  OOD: {acc_ood_t:.1%}  gap: {gap_t:.1%}')

print('\n=== Tilted ERM 捷径检测 ===')
shortcut_detection(model_tilted, id_dataset, device=DEVICE)